# MeatGuard-S — Training & Benchmarking

**Phase 1 · RO1** — Train the lightweight hybrid CNN–Transformer **MeatGuard-S** (~3.4M params) on the MeatScan-style dataset with strict **benchmark parity** (same ingestion, augmentation, 70/15/15 split, AdamW + cosine LR, class-weighted CE).

Mirrors `src/model/trainer.py`, `src/model/architecture.py`, `src/data/`.

## 1. Setup & installation

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Run once on a fresh Colab/Kaggle runtime (uncomment):
# !pip -q install torch torchvision timm einops scikit-learn matplotlib seaborn tqdm opencv-python shap scikit-image onnx

In [ ]:
import os, sys

# --- Path bootstrap: make ../src importable on local, Colab, and Kaggle ------
def add_src_to_path():
    here = os.getcwd()
    candidates = [
        os.path.abspath(os.path.join(here, "..", "src")),   # local: notebooks/ -> ../src
        os.path.abspath(os.path.join(here, "src")),         # run from repo root
        "/content/drive/MyDrive/MeatGuard_II",                        # Colab (git clone target)
        "/kaggle/working/MeatGuard_II/src",                 # Kaggle
    ]
    for c in candidates:
        if os.path.isdir(c):
            if c not in sys.path:
                sys.path.insert(0, c)
            return c
    raise FileNotFoundError(
        "Could not locate the MeatGuard_II/src package. "
        "On Colab/Kaggle, clone the repo into /content (or /kaggle/working) first."
    )

SRC_DIR = add_src_to_path()
print("Using src package at:", SRC_DIR)
os.chdir(SRC_DIR)
print(f"Changed working directory to: {os.getcwd()}")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from src.data.dataset import make_loaders
from src.model.architecture import MeatGuardS
from src.model.trainer import MeatGuardTrainer

## 2. Configuration

In [ ]:
# Placeholder dataset (drop-in stand-in for the full MeatScan set).
# Override with:  os.environ["MEATGUARD_DATA"] = "/path/to/your/data"
DATA_DIR = os.environ.get("MEATGUARD_DATA", r"D:/Research_Work/MeatGuard/data/test")
ARTIFACTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "artifacts"))
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
print("Data dir :", DATA_DIR)
print("Artifacts:", ARTIFACTS_DIR)

In [ ]:
CONFIG = {
    "strategy":     "domain",   # baseline | standard | autoaugment | randaugment | cutmix_mixup | domain
    "batch_size":   32,
    "epochs":       50,
    "lr":           1e-3,
    "weight_decay": 1e-4,
    "seed":         42,
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
}
print(CONFIG)

## 3. Data ingestion (benchmark parity)

`make_loaders` returns train/val/test loaders plus inverse-frequency class weights. Val/test always use the clean (no-augmentation) transform.

In [ ]:
train_loader, val_loader, test_loader, class_weights = make_loaders(
    DATA_DIR,
    strategy=CONFIG["strategy"],
    batch_size=CONFIG["batch_size"],
    seed=CONFIG["seed"],
)
print(f"train batches: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")
print("class weights:", class_weights.tolist())

## 4. Model initialisation

In [ ]:
model = MeatGuardS(num_classes=2)
n_params = sum(p.numel() for p in model.parameters())
print(f"MeatGuard-S parameters: {n_params/1e6:.2f}M")

criterion = nn.CrossEntropyLoss(weight=class_weights.to(CONFIG["device"]))
optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)

## 5. Training loop

In [ ]:
trainer = MeatGuardTrainer(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device=CONFIG["device"],
)

history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_val = float("inf")

for epoch in tqdm(range(CONFIG["epochs"]), desc="Training Progress"):
    train_loss = trainer.train_epoch()
    val_loss, val_acc = trainer.validate()
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), os.path.join(ARTIFACTS_DIR, "meatguard_s_best.pth"))

    print(f"Epoch {epoch+1:02d}/{CONFIG['epochs']} | "
          f"train {train_loss:.4f} | val {val_loss:.4f} | val_acc {val_acc:.4f}")

In [ ]:
# Training curves
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["val_acc"], color="green")
ax[1].set_title("Validation accuracy"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 6. Test evaluation (RO1 metrics)

Reports accuracy, macro-F1, and the food-safety-critical **False Negative Rate** (spoiled predicted as fresh). Class index convention: `Fresh=0`, `Spoiled=1`.

In [ ]:
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in tqdm(
        test_loader, desc="Testing"):
        logits = model(images.to(CONFIG["device"]))
        preds = logits.argmax(1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(labels.numpy().tolist())

y_true, y_pred = np.array(y_true), np.array(y_pred)
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
fnr = fn / (fn + tp) if (fn + tp) else float("nan")

print(classification_report(y_true, y_pred,
      target_names=["Fresh", "Spoiled"], zero_division=0))
print(f"Macro-F1 : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"FNR      : {fnr:.4f}  (spoiled misclassified as fresh)")
print("Confusion matrix [ [TN FP] [FN TP] ]:\n", cm)

## 7. Save final checkpoint

In [ ]:
final_path = os.path.join(ARTIFACTS_DIR, "meatguard_s_final.pth")
torch.save(model.state_dict(), final_path)
print("Saved ->", final_path)